In [15]:
import requests
import pandas as pd
import re
import time
from bs4 import BeautifulSoup
import io
import numpy as np
from tqdm import tqdm

In [16]:
url = "https://www.basketball-reference.com/teams/"
headers= {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers, timeout=20)
print(response.status_code)
soup = BeautifulSoup(response.text, "html.parser")

200


In [17]:
team_ids = ['ATL','BOS','NJN','CHA','CHI','CLE','DAL','DEN','DET','GSW','HOU','IND','LAC','LAL','MEM',
 'MIA','MIL','MIN','NOH','NYK','OKC','ORL','PHI','PHO','POR','SAC','SAS','TOR','UTA','WAS']

years = [2020, 2021, 2022, 2023, 2024, 2025, 2026]

In [22]:
def get_arena(team_id, year):

    url = f"https://www.basketball-reference.com/teams/{team_id}/{year}.html"

    arena = np.nan
    attendance = np.nan
    attendance_rank = np.nan

    try:

        response = requests.get(url, headers=headers, timeout=20)

        if response.status_code != 200:

            return {
                "team_id": team_id,
                "year": year,
                "arena": arena,
                "attendance": attendance,
                "attendance_rank": attendance_rank
            }

        soup = BeautifulSoup(response.text, "html.parser")

        paragraphs = soup.find_all("p")

        for p in paragraphs:

            text = p.get_text(" ", strip=True)

            if "Arena:" in text:

                arena = (text.split("Arena:")[1].split("Attendance:")[0].strip())

                if "Attendance:" in text:

                    attendance = (text.split("Attendance:")[1].split("(")[0].strip().replace(",", ""))

                if "(" in text and ")" in text:

                    attendance_rank = (
                        text.split("(")[1]
                        .split(")")[0]
                        .strip()
                    )

                break

    except:
        pass

    return {
        "team_id": team_id,
        "year": year,
        "arena": arena,
        "attendance": attendance,
        "attendance_rank": attendance_rank
    }

In [23]:
arena_data = []

for team_id in tqdm(team_ids):

    for year in years:

        result = get_arena(team_id, year)

        arena_data.append(result)

        time.sleep(5)

arena = pd.DataFrame(arena_data)

100%|██████████| 30/30 [21:24<00:00, 42.83s/it]


In [24]:
arena

,team_id,year,arena,attendance,attendance_rank
0,ATL,2020,State Farm Arena,545453,20th of 30
1,ATL,2021,State Farm Arena,59288,13th of 30
2,ATL,2022,State Farm Arena,672742,17th of 30
3,ATL,2023,State Farm Arena,719787,18th of 30
4,ATL,2024,State Farm Arena,696418,25th of 30
...,...,...,...,...,...
205,WAS,2022,Capital One Arena,637215,23rd of 30
206,WAS,2023,Capital One Arena,710481,21st of 30
207,WAS,2024,Capital One Arena,692851,26th of 30
208,WAS,2025,Capital One Arena,663691,29th of 30


In [ ]:
arena.to_csv("arena.csv", index=False, encoding="utf-8")